# PySR Symbolic Regression — Aquaculture Water Quality

**How to use:** Run each cell one at a time by clicking it and pressing **Shift+Enter**.
Each model cell is independent — PySR will print live progress in the output below the cell.
You can stop any model early by pressing **'q' + Enter** in the output box; it will return the best formula found so far.

---
| Model | Target | Inputs |
|-------|--------|--------|
| 1 | DO (No Lag) | temp, ph, turbidity, tds |
| 2 | DO (With Lag) | do_lag1, temp, ph, turbidity, tds |
| 3 | TDS | tds_lag1, temp, ph, turbidity |
| 4 | pH | ph_lag1, temp, turbidity, tds |
| 5 | TEMP | temp_lag1, ph, turbidity, tds |
| 6 | TURBIDITY | turbidity_lag1, temp, ph, tds |

---

## Step 1 — Import Libraries
Run this cell first. It loads all required packages.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pysr import PySRRegressor
import warnings
warnings.filterwarnings('ignore')

print('✓ Libraries loaded successfully.')

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
✓ Libraries loaded successfully.


## Step 2 — Load Dataset & Create Lag Features
Loads the CSV, standardises column names, and creates the t-1 lag features BEFORE removing outliers to preserve time-series continuity.

In [3]:
# Load dataset (must be in the same folder as this notebook)
df = pd.read_csv('Aquaculture_Master_Dataset_with_TDS.csv')

# Standardise column names: lowercase, replace special chars with underscore
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9_]+', '_', regex=True)
    .str.strip('_')
)

print(f'Raw dataset shape  : {df.shape}')

# Keep only the 5 target columns
required_cols = ['do', 'temp', 'ph', 'turbidity', 'tds']

df = df[required_cols].copy()

# Convert to numeric, interpolate gaps
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.interpolate(method='linear', limit_direction='both')

# Create lag features BEFORE dropping rows to preserve time-series sequence
for col in required_cols:
    df[f'{col}_lag1'] = df[col].shift(1)

# Drop first row which now has NaNs
df = df.dropna().reset_index(drop=True)

print(f'\nShape after adding lags: {df.shape}')
print(f'Columns: {list(df.columns)}')

Raw dataset shape  : (74758, 12)

Shape after adding lags: (74757, 10)
Columns: ['do', 'temp', 'ph', 'turbidity', 'tds', 'do_lag1', 'temp_lag1', 'ph_lag1', 'turbidity_lag1', 'tds_lag1']


## Step 3 — Remove Outliers (IQR method)
Uses the Interquartile Range to remove extreme values from each column.

In [4]:
original_rows = len(df)
mask = pd.Series([True] * original_rows, index=df.index)

print('Outlier removal per column (IQR x1.5):')
for col in required_cols: # Calculate IQR only on base columns
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    col_mask = (df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)
    removed = (~col_mask).sum()
    print(f'  {col:<12}: {removed} outliers removed')
    mask = mask & col_mask

df = df[mask].reset_index(drop=True)

print(f'\nOriginal rows : {original_rows}')
print(f'After cleanup : {len(df)}  ({(original_rows - len(df)) / original_rows * 100:.1f}% removed)')
print('\n✓ Dataset ready for PySR training.')

Outlier removal per column (IQR x1.5):
  do          : 0 outliers removed
  temp        : 2120 outliers removed
  ph          : 0 outliers removed
  turbidity   : 2210 outliers removed
  tds         : 10843 outliers removed

Original rows : 74757
After cleanup : 61059  (18.3% removed)

✓ Dataset ready for PySR training.


---
## Model 1 — DO (No Lag)
**Target:** `do`  
**Inputs:** `['temp', 'ph', 'turbidity', 'tds']`

> ⏱ Expected run time: 10–30 min depending on your CPU. Stop early with **q + Enter**.

In [9]:
print('=' * 60)
print('TRAINING MODEL 1 — DO (NO LAG)')
print('=' * 60)

target_col  = 'do'
input_cols  = ['temp', 'ph', 'turbidity', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_do_nolag = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['sqrt', 'log'],
    maxsize=25,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.002,
)
model_do_nolag.fit(X_train, y_train)

y_pred_train = model_do_nolag.predict(X_train)
y_pred_test  = model_do_nolag.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 1 — DO (No Lag)')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  do = {model_do_nolag.sympy()}')
print('=' * 60)

Compiling Julia backend...


TRAINING MODEL 1 — DO (NO LAG)
Training samples : 48847
Test samples     : 12212
Target mean±std  : 10.627 ± 4.970

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 2.770e+03
Progress: 57 / 10000 total iterations (0.570%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.466e+01  0.000e+00  y = 10.653
3           2.337e+01  2.685e-02  y = ph - -4.021
6           2.336e+01  9.651e-05  y = sqrt(tds / temp) + ph
9           2.287e+01  7.085e-03  y = ph + ((turbidity / (temp + -5.6036)) - -2.6302)
11          2.284e+01  8.050e-04  y = (ph + ((turbidity / (temp + -6.2706)) - -0.41295)) - -...
                                      2.0148
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per se

[ Info: Final population:
[ Info: Results saved to:



RESULTS: Model 1 — DO (No Lag)
R² (Train) : 0.0864
R² (Test)  : 0.0788
MSE        : 22.9165
RMSE       : 4.7871
MAE        : 3.8784

Best Formula:
  do = ph + 103.36679/temp


---
## Model 2 — DO (With Lag)
**Target:** `do`  
**Inputs:** `['do_lag1', 'temp', 'ph', 'turbidity', 'tds']`

> ⏱ Expected run time: 10–30 min depending on your CPU. Stop early with **q + Enter**.

In [10]:
print('=' * 60)
print('TRAINING MODEL 2 — DO (WITH LAG)')
print('=' * 60)

target_col  = 'do'
input_cols  = ['do_lag1', 'temp', 'ph', 'turbidity', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_do_lag = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['sqrt', 'log'],
    maxsize=25,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.002,
)
model_do_lag.fit(X_train, y_train)

y_pred_train = model_do_lag.predict(X_train)
y_pred_test  = model_do_lag.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 2 — DO (With Lag)')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  do = {model_do_lag.sympy()}')
print('=' * 60)

TRAINING MODEL 2 — DO (WITH LAG)
Training samples : 48847
Test samples     : 12212
Target mean±std  : 10.627 ± 4.970

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 2.130e+03
Progress: 43 / 10000 total iterations (0.430%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.466e+01  0.000e+00  y = 10.606
3           2.457e+01  1.919e-03  y = ph / 0.5803
4           2.378e+01  3.260e-02  y = sqrt(do_lag1 + 101.48)
5           2.065e+01  1.410e-01  y = (do_lag1 * 0.40383) - -6.3574
7           2.031e+01  8.349e-03  y = (ph - (do_lag1 * -0.36372)) + 0.18426
8           1.982e+01  2.438e-02  y = ph + ((log(do_lag1) + -1.2071) * 3.8983)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per secon

[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.210e+03
Progress: 9791 / 10000 total iterations (97.910%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.466e+01  0.000e+00  y = 10.629
3           2.337e+01  2.684e-02  y = ph + 4.0139
4           2.004e+01  1.536e-01  y = log(do_lag1) * 4.7092
6           2.000e+01  1.131e-03  y = ph + (do_lag1 / sqrt(ph))
7           1.935e+01  3.281e-02  y = (-124.69 / (ph + do_lag1)) + 18.551
9           1.931e+01  1.106e-03  y = (-84.194 / ((do_lag1 + ph) + -2.6515)) + 17.193
10          1.898e+01  1.737e-02  y = ((log(do_lag1) + -2.925) * (temp / ph)) + 13.584
12          1.897e+01  7.052e-05  y = ((log(do_lag1) + -2.925) * ((temp / ph) + -0.04558)) +...
                                       13.584
13          1.883e+01  7.878e-03  y = ((log(do_lag1) 

---
## Model 3 — TDS
**Target:** `tds`  
**Inputs:** `['tds_lag1', 'temp', 'ph', 'turbidity']`

> ⏱ Expected run time: 10–30 min depending on your CPU. Stop early with **q + Enter**.

In [11]:
print('=' * 60)
print('TRAINING MODEL 3 — TDS')
print('=' * 60)

target_col  = 'tds'
input_cols  = ['tds_lag1', 'temp', 'ph', 'turbidity']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_tds = PySRRegressor(
    niterations=300,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=[],
    maxsize=12,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.01,
)
model_tds.fit(X_train, y_train)

y_pred_train = model_tds.predict(X_train)
y_pred_test  = model_tds.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 3 — TDS')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  tds = {model_tds.sympy()}')
print('=' * 60)

TRAINING MODEL 3 — TDS
Training samples : 48847
Test samples     : 12212
Target mean±std  : 517.694 ± 199.229

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 4.970e+03
Progress: 105 / 6000 total iterations (1.750%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           3.963e+04  0.000e+00  y = 517.21
3           3.902e+04  5.709e-03  y = turbidity + 486.84
5           3.084e+04  1.163e-01  y = (tds_lag1 * 0.28641) - -354.76
7           3.061e+04  2.782e-03  y = (turbidity + (tds_lag1 * 0.28061)) + 328.99
9           3.055e+04  -0.000e+00  y = (turbidity * 1.7822) - ((tds_lag1 * -0.27435) + -307....
                                       66)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evalua

[ Info: Final population:
[ Info: Results saved to:



RESULTS: Model 3 — TDS
R² (Train) : 0.2217
R² (Test)  : 0.2291
MSE        : 30796.1305
RMSE       : 175.4883
MAE        : 138.3769

Best Formula:
  tds = tds_lag1*0.29160655 + 352.136
  - outputs\20260510_173554_2L51ee\hall_of_fame.csv


---
## Model 4 — pH
**Target:** `ph`  
**Inputs:** `['ph_lag1', 'temp', 'turbidity', 'tds']`

> ⏱ Expected run time: 10–30 min depending on your CPU. Stop early with **q + Enter**.

In [12]:
print('=' * 60)
print('TRAINING MODEL 4 — PH')
print('=' * 60)

target_col  = 'ph'
input_cols  = ['ph_lag1', 'temp', 'turbidity', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_ph = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['log', 'sqrt'],
    maxsize=20,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.003,
)
model_ph.fit(X_train, y_train)

y_pred_train = model_ph.predict(X_train)
y_pred_test  = model_ph.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 4 — pH')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  ph = {model_ph.sympy()}')
print('=' * 60)

TRAINING MODEL 4 — PH
Training samples : 48847
Test samples     : 12212
Target mean±std  : 6.617 ± 1.108

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 2.570e+03
Progress: 53 / 10000 total iterations (0.530%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.227e+00  0.000e+00  y = 6.6174
4           1.156e+00  1.969e-02  y = sqrt(ph_lag1 + 37.099)
5           1.013e+00  1.322e-01  y = 3.8125 - (ph_lag1 * -0.42358)
10          1.007e+00  1.183e-03  y = log(((ph_lag1 + 1.2847) * ph_lag1) * ph_lag1) - -0.805...
                                      36
13          1.004e+00  1.181e-03  y = log(ph_lag1 * ((ph_lag1 + (ph_lag1 / sqrt(temp))) * ph...
                                      _lag1)) - -0.80536
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════

[ Info: Final population:
[ Info: Results saved to:



RESULTS: Model 4 — pH
R² (Train) : 0.1710
R² (Test)  : 0.1643
MSE        : 1.0259
RMSE       : 1.0128
MAE        : 0.8179

Best Formula:
  ph = 2.5795584*sqrt(ph_lag1)
  - outputs\20260510_174133_NLaKfx\hall_of_fame.csv


---
## Model 5 — TEMP
**Target:** `temp`  
**Inputs:** `['temp_lag1', 'ph', 'turbidity', 'tds']`

> ⏱ Expected run time: 10–30 min depending on your CPU. Stop early with **q + Enter**.

In [5]:
print('=' * 60)
print('TRAINING MODEL 5 — TEMP')
print('=' * 60)

target_col  = 'temp'
input_cols  = ['temp_lag1', 'ph', 'turbidity', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_temp = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['log'],
    maxsize=20,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.003,
)
model_temp.fit(X_train, y_train)

y_pred_train = model_temp.predict(X_train)
y_pred_test  = model_temp.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 5 — TEMP')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  temp = {model_temp.sympy()}')
print('=' * 60)

Compiling Julia backend...


TRAINING MODEL 5 — TEMP
Training samples : 48847
Test samples     : 12212
Target mean±std  : 26.517 ± 4.959

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 2.030e+03
Progress: 42 / 10000 total iterations (0.420%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.469e+01  0.000e+00  y = 26.522
5           2.078e+01  4.309e-02  y = (temp_lag1 * 0.38305) + 16.203
8           2.069e+01  1.413e-03  y = (log(tds) + (temp_lag1 * 0.35547)) + 10.828
10          2.069e+01  2.235e-06  y = (((temp_lag1 * 0.35554) + 6.925) + log(tds)) - -3.913
16          2.067e+01  1.245e-04  y = (((log(turbidity + tds) + 5.0771) + (temp_lag1 * 0.355...
                                      54)) - -3.3932) + (2.3556 - 0.086142)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Pre

[ Info: Final population:
[ Info: Results saved to:



RESULTS: Model 5 — TEMP
R² (Train) : 0.1417
R² (Test)  : 0.1405
MSE        : 20.8085
RMSE       : 4.5616
MAE        : 3.6181

Best Formula:
  temp = 8.1226*log(temp_lag1)
  - outputs\20260510_181223_ePYcI7\hall_of_fame.csv


---
## Model 6 — TURBIDITY
**Target:** `turbidity`  
**Inputs:** `['turbidity_lag1', 'temp', 'ph', 'tds']`

> ⏱ Expected run time: 10–30 min depending on your CPU. Stop early with **q + Enter**.

In [6]:
print('=' * 60)
print('TRAINING MODEL 6 — TURBIDITY')
print('=' * 60)

target_col  = 'turbidity'
input_cols  = ['turbidity_lag1', 'temp', 'ph', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_turbidity = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['log', 'sqrt'],
    maxsize=22,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.002,
)
model_turbidity.fit(X_train, y_train)

y_pred_train = model_turbidity.predict(X_train)
y_pred_test  = model_turbidity.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 6 — TURBIDITY')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  turbidity = {model_turbidity.sympy()}')
print('=' * 60)

TRAINING MODEL 6 — TURBIDITY
Training samples : 48847
Test samples     : 12212
Target mean±std  : 29.685 ± 9.437

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 1.770e+03
Progress: 37 / 10000 total iterations (0.370%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           8.897e+01  0.000e+00  y = 29.64
4           8.511e+01  1.478e-02  y = sqrt(turbidity_lag1) - -24.108
11          8.460e+01  8.627e-04  y = sqrt((ph + 8.8657) * (sqrt(tds) + (turbidity_lag1 - -0...
                                      .82854)))
13          8.321e+01  8.281e-03  y = sqrt(((2.5829 / 0.29134) + ph) * (turbidity_lag1 + sqr...
                                      t(tds))) + 1.4789
14          8.141e+01  2.185e-02  y = sqrt((((sqrt(temp) + (turbidity_lag1 + temp)) + turbid...
                                      ity_lag1) + sqrt(temp)) * 9.0743)
15          8.011e+01  1.601e-02  y = (((temp + (tds * 0.012946)) * log(turbidity_lag1)

[ Info: Final population:
[ Info: Results saved to:



RESULTS: Model 6 — TURBIDITY
R² (Train) : 0.0582
R² (Test)  : 0.0554
MSE        : 84.4248
RMSE       : 9.1883
MAE        : 7.4741

Best Formula:
  turbidity = 8.84823*log(turbidity_lag1)


---
## Step 4 — Summary: All Formulas
Run this cell AFTER all model cells above have finished to print all formulas together.

In [ ]:
print('\n' + '#' * 60)
print('#' + ' ' * 58 + '#')
print('#' + '       ALL DERIVED FORMULAS — AQUACULTURE WQMS        ' + '#')
print('#' + ' ' * 58 + '#')
print('#' * 60)

formulas = {
    'DO (No Lag)  (inputs: temp, ph, turbidity, tds)       ': model_do_nolag,
    'DO (Lag)     (inputs: do_lag1, temp, ph, turbidity, tds)': model_do_lag,
    'TDS          (inputs: tds_lag1, temp, ph, turbidity)   ': model_tds,
    'pH           (inputs: ph_lag1, temp, turbidity, tds)   ': model_ph,
    'TEMP         (inputs: temp_lag1, ph, turbidity, tds)   ': model_temp,
    'TURBIDITY    (inputs: turbidity_lag1, temp, ph, tds)   ': model_turbidity,
}

for name, model in formulas.items():
    print(f'\n{name}')
    try:
        print(f'  Formula : {model.sympy()}')
    except Exception as e:
        print(f'  Formula : [Not trained yet]')

print('\n' + '#' * 60)

In [7]:
print('=' * 60)
print('TRAINING MODEL 6 — TURBIDITYNOLAG')
print('=' * 60)

target_col  = 'turbidity'
input_cols  = ['temp', 'ph', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_turbidity = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['log', 'sqrt'],
    maxsize=22,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.002,
)
model_turbidity.fit(X_train, y_train)

y_pred_train = model_turbidity.predict(X_train)
y_pred_test  = model_turbidity.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 6 — TURBIDITY')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  turbidity = {model_turbidity.sympy()}')
print('=' * 60)

TRAINING MODEL 6 — TURBIDITYNOLAG
Training samples : 48847
Test samples     : 12212
Target mean±std  : 29.685 ± 9.437

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 1.750e+03
Progress: 35 / 10000 total iterations (0.350%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           8.897e+01  0.000e+00  y = 29.649
6           8.816e+01  1.826e-03  y = sqrt((temp + tds) - -354.08)
7           8.590e+01  2.591e-02  y = ((temp * 0.0072466) * temp) + 24.223
12          8.464e+01  2.969e-03  y = log(((tds * tds) * tds) * tds) + (temp / ph)
14          8.394e+01  4.146e-03  y = (log((tds * tds) * (tds * tds)) + (temp / ph)) - -1.02...
                                      69
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Exp

[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.890e+03
Progress: 9981 / 10000 total iterations (99.810%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           8.897e+01  0.000e+00  y = 29.698
3           8.739e+01  8.911e-03  y = 36.319 - ph
4           8.652e+01  9.986e-03  y = log(tds) * 4.8073
5           8.593e+01  6.919e-03  y = (tds * 0.0087776) + 25.145
6           8.490e+01  1.201e-02  y = sqrt(tds / ph) + 20.928
7           8.428e+01  7.290e-03  y = ((484 / temp) + temp) + -15.739
8           8.421e+01  8.152e-04  y = ((sqrt(tds) + 46.04) + temp) / 3.1954
9           8.344e+01  9.246e-03  y = ((temp / ph) - (tds * -0.0080701)) + 21.377
11          8.237e+01  6.422e-03  y = (((tds * 0.0068961) + temp) + -19.478) + (488.24 / tem...
                                      p)
12          8.196e

In [8]:
print('=' * 60)
print('TRAINING MODEL 4 — PHNOLAG')
print('=' * 60)

target_col  = 'ph'
input_cols  = ['temp', 'turbidity', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_ph = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['log', 'sqrt'],
    maxsize=20,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.003,
)
model_ph.fit(X_train, y_train)

y_pred_train = model_ph.predict(X_train)
y_pred_test  = model_ph.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 4 — pH')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  ph = {model_ph.sympy()}')
print('=' * 60)

TRAINING MODEL 4 — PHNOLAG
Training samples : 48847
Test samples     : 12212
Target mean±std  : 6.617 ± 1.108

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 2.240e+03
Progress: 45 / 10000 total iterations (0.450%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.227e+00  0.000e+00  y = 6.6176
5           1.226e+00  2.572e-04  y = 6.6176 - (-0.40453 / temp)
6           1.156e+00  5.847e-02  y = (sqrt(temp) * -0.58265) + 9.5953
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 1.710e+03
Progress: 69 / 10000 total iterations (0.690%)
════════════════════════════════════════════════════════════════════════════════════════════════════
─────────────────────────────────────

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.227e+00  0.000e+00  y = 6.6175
4           1.163e+00  1.773e-02  y = 9.8777 - log(temp)
5           1.154e+00  8.016e-03  y = (temp * -0.054315) + 8.0579
6           1.154e+00  2.961e-04  y = sqrt((temp * -0.71776) - -62.894)
7           1.153e+00  5.437e-04  y = (temp * (temp * -0.0010028)) + 7.3475
8           1.148e+00  4.726e-03  y = ((temp + sqrt(turbidity)) * -0.056511) + 8.4253
9           1.136e+00  1.023e-02  y = ((turbidity * -0.045745) + 8.9781) - (temp / turbidity...
                                      )
11          1.132e+00  1.603e-03  y = (((-0.58742 / turbidity) + -0.03033) * (turbidity + te...
                                      mp)) + 9.4823
13          1.131e+00  7.683e-04  y = (((turbidity + temp) / log(sqrt(turbidity + -5.8041)))...
                                       * -0.081865) + 9.594
15          1.1

In [9]:
print('=' * 60)
print('TRAINING MODEL 5 — TEMPNOLAG')
print('=' * 60)

target_col  = 'temp'
input_cols  = ['ph', 'turbidity', 'tds']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_temp = PySRRegressor(
    niterations=500,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['log'],
    maxsize=20,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.003,
)
model_temp.fit(X_train, y_train)

y_pred_train = model_temp.predict(X_train)
y_pred_test  = model_temp.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 5 — TEMP')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  temp = {model_temp.sympy()}')
print('=' * 60)

TRAINING MODEL 5 — TEMPNOLAG
Training samples : 48847
Test samples     : 12212
Target mean±std  : 26.517 ± 4.959

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 1.400e+03
Progress: 29 / 10000 total iterations (0.290%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.469e+01  0.000e+00  y = 26.593
7           2.399e+01  4.831e-03  y = ((turbidity * 0.067677) + 16.796) * 1.4084
9           2.323e+01  1.601e-02  y = ph + (((ph * 105.58) / -49.708) - -33.857)
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 1.390e+03
Progress: 58 / 10000 total iterations (0.580%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────

[ Info: Final population:
[ Info: Results saved to:


───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.469e+01  0.000e+00  y = 26.514
3           2.323e+01  3.037e-02  y = 33.131 - ph
5           2.322e+01  2.252e-04  y = 33.692 - (ph * 1.0848)
6           2.296e+01  1.129e-02  y = log(turbidity) + (29.791 - ph)
7           2.276e+01  8.985e-03  y = ((turbidity / 13.57) - ph) + 30.941
8           2.275e+01  1.446e-04  y = (log(tds * turbidity) - ph) + 23.617
9           2.264e+01  5.085e-03  y = (turbidity * (turbidity * 0.0012202)) + (31.948 - ph)
10          2.260e+01  1.814e-03  y = (log(tds) + ((turbidity / 15.188) - ph)) + 25.007
11          2.251e+01  3.856e-03  y = (((turbidity / tds) * (turbidity + -43.187)) - ph) + 3...
                                      3.868
13          2.243e+01  1.716e-03  y = (((turbidity + -42.843) * (turbidity / tds)) / 0.70461...
                                      ) + (34.146 - ph)
15         

In [10]:
print('=' * 60)
print('TRAINING MODEL 3 — TDSNOLAG')
print('=' * 60)

target_col  = 'tds'
input_cols  = ['temp', 'ph', 'turbidity']

X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Target mean±std  : {y.mean():.3f} ± {y.std():.3f}')
print('\nStarting PySR...\n')

model_tds = PySRRegressor(
    niterations=300,
    populations=20,
    tournament_selection_n=10,
    batching=True,
    batch_size=8000,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=[],
    maxsize=12,
    elementwise_loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    random_state=42,
    deterministic=True,
    parallelism='serial',
    verbosity=1,
    parsimony=0.01,
)
model_tds.fit(X_train, y_train)

y_pred_train = model_tds.predict(X_train)
y_pred_test  = model_tds.predict(X_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)
mse_test  = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)

print('\n' + '=' * 60)
print('RESULTS: Model 3 — TDS')
print('=' * 60)
print(f'R² (Train) : {r2_train:.4f}')
print(f'R² (Test)  : {r2_test:.4f}')
print(f'MSE        : {mse_test:.4f}')
print(f'RMSE       : {rmse_test:.4f}')
print(f'MAE        : {mae_test:.4f}')
print(f'\nBest Formula:')
print(f'  tds = {model_tds.sympy()}')
print('=' * 60)

TRAINING MODEL 3 — TDSNOLAG
Training samples : 48847
Test samples     : 12212
Target mean±std  : 517.694 ± 199.229

Starting PySR...



[ Info: Started!



Expressions evaluated per second: 4.520e+03
Progress: 96 / 6000 total iterations (1.600%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           3.963e+04  0.000e+00  y = 517.35
3           3.902e+04  5.709e-03  y = turbidity + 488.12
5           3.827e+04  7.701e-03  y = 401.89 - (turbidity * -3.8766)
9           3.824e+04  -0.000e+00  y = (((turbidity + -6.2197) * 4.0308) + 428.97) - ph
11          3.811e+04  -0.000e+00  y = (temp / ph) + ((temp - (turbidity * -4.0456)) + 363.4...
                                       )
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions eva

[ Info: Final population:
[ Info: Results saved to:



RESULTS: Model 3 — TDS
R² (Train) : 0.0635
R² (Test)  : 0.0626
MSE        : 37449.1708
RMSE       : 193.5179
MAE        : 147.2476

Best Formula:
  tds = temp*(temp - 1*49.561565) - 1*(-1104.1257)
  - outputs\20260510_202906_ihgFvx\hall_of_fame.csv
